# Final Test — Independent Ground Truth + Precision/Recall

Notebook ini mengevaluasi konfigurasi **frozen** terhadap
`ground_truth_relevance_test_v2.csv`.

Konfigurasi tidak diubah:

- Query: `query_text_v2` / BASE_V2
- Semantic retrieval: Top-50
- Keyword retrieval tambahan: maksimal 20
- Semantic normalization: min-max per candidate pool
- WLC: `0.7 × semantic + 0.3 × harmonic efficiency`
- Test set: 90 query

Metrik:

- Candidate Hit Rate
- Candidate Recall
- Hit@5
- Hit@10
- Precision@5
- Precision@10
- Recall@5
- Recall@10
- MRR

Catatan: Recall dihitung terhadap **relevance set yang telah ditetapkan**, bukan terhadap
seluruh resep relevan yang mungkin ada di corpus.


In [1]:
import pandas as pd
import numpy as np

from dapuria_pipeline import (
    DapuriaRecipeService,
    parse_user_ingredients,
    coverage_stats,
    harmonic_efficiency,
    TOP_K_SBERT,
    MAX_KEYWORD_EXTRA,
    W1_FINAL,
    W2_FINAL,
)

TEST_PATH = "ground_truth_relevance_test_v2.csv"

SUMMARY_OUT = "hasil_final_test_independent_groundtruth_precision_recall_ringkas.csv"
DETAIL_OUT = "hasil_final_test_independent_groundtruth_precision_recall_detail.csv"


def split_pipe(x):
    if pd.isna(x):
        return []
    return [s.strip() for s in str(x).split(" | ") if s.strip()]


def precision_at_k(ranked_ids, relevant_ids, k):
    topk = ranked_ids[:k]
    if k <= 0:
        return 0.0
    n_rel = sum(1 for rid in topk if rid in relevant_ids)
    return n_rel / k


def recall_at_k(ranked_ids, relevant_ids, k):
    if not relevant_ids:
        return 0.0
    topk = ranked_ids[:k]
    n_rel = sum(1 for rid in topk if rid in relevant_ids)
    return n_rel / len(relevant_ids)


def candidate_recall(candidate_ids, relevant_ids):
    if not relevant_ids:
        return 0.0
    n_rel = sum(1 for rid in candidate_ids if rid in relevant_ids)
    return n_rel / len(relevant_ids)


def hit_at_k(ranked_ids, relevant_ids, k):
    return int(any(rid in relevant_ids for rid in ranked_ids[:k]))


def reciprocal_rank(ranked_ids, relevant_ids):
    for i, rid in enumerate(ranked_ids, start=1):
        if rid in relevant_ids:
            return 1.0 / i
    return 0.0


# ============================================================
# 1. Load test set
# ============================================================
test = pd.read_csv(TEST_PATH)

required_cols = {
    "scraped_title",
    "query_text_v2",
    "relevant_ids",
}

missing = required_cols - set(test.columns)
if missing:
    raise ValueError(f"Kolom test hilang: {sorted(missing)}")

dup_cases = test.duplicated(["query_text_v2", "relevant_ids"]).sum()

if dup_cases:
    raise ValueError(f"Test masih punya {dup_cases} duplicate query+relevance cases.")

print("VALIDASI TEST SET")
print(f"Jumlah query              : {len(test)}")
print(f"Duplicate query+relevance : {dup_cases}")


# ============================================================
# 2. Frozen production service
# ============================================================
service = DapuriaRecipeService(
    chroma_dir="chroma_db",
    collection_name="resep_masakan",
)

print("\nFROZEN CONFIG")
print(f"Semantic top-k            : {TOP_K_SBERT}")
print(f"Keyword extra max         : {MAX_KEYWORD_EXTRA}")
print(
    f"WLC                       : {W1_FINAL:.1f} semantic + {W2_FINAL:.1f} efficiency"
)
print("Semantic normalization    : min-max")
print("Efficiency                : harmonic mean")


# ============================================================
# 3. Validate all relevant IDs
# ============================================================
all_relevant_ids = sorted(
    {rid for ids in test["relevant_ids"].apply(split_pipe) for rid in ids}
)

found = service.collection.get(
    ids=all_relevant_ids,
    include=[],
)

found_ids = set(found["ids"])
missing_ids = [rid for rid in all_relevant_ids if rid not in found_ids]

print("\nGROUND TRUTH VALIDATION")
print(f"Unique relevant IDs          : {len(all_relevant_ids)}")
print(f"Relevant IDs found in Chroma : {len(found_ids)}")

if missing_ids:
    raise ValueError(
        f"Ada {len(missing_ids)} relevant ID yang tidak tersedia di Chroma. "
        f"Contoh: {missing_ids[:10]}"
    )


# ============================================================
# 4. Evaluate frozen configuration
# ============================================================
rows = []

print("\nMenjalankan frozen configuration pada test set...")

for _, row in test.iterrows():
    query_text = str(row["query_text_v2"]).strip()
    relevant_ids = set(split_pipe(row["relevant_ids"]))
    user_ingredients = parse_user_ingredients(query_text)

    # Candidate pool
    candidates = service.retrieve_candidates(
        query_text,
        user_ingredients,
    )

    candidate_ids = [str(c["id"]) for c in candidates]

    # Candidate metrics
    cand_hit = int(any(rid in relevant_ids for rid in candidate_ids))
    cand_recall = candidate_recall(
        candidate_ids,
        relevant_ids,
    )

    # Frozen reranking
    if candidates:
        semantic_scores = np.asarray(
            [float(c["sbert_score"]) for c in candidates],
            dtype=float,
        )

        s_min = float(semantic_scores.min())
        s_max = float(semantic_scores.max())

        semantic_norm = (semantic_scores - s_min) / (s_max - s_min + 1e-9)

        ranked = []

        for c, s_norm in zip(candidates, semantic_norm):
            stats = coverage_stats(
                user_ingredients,
                c["ingredient_names"],
            )

            efficiency = harmonic_efficiency(
                stats["coverage_recipe"],
                stats["coverage_user"],
            )

            final_score = W1_FINAL * float(s_norm) + W2_FINAL * float(efficiency)

            ranked.append(
                {
                    **c,
                    "sbert_score_norm": float(s_norm),
                    "efficiency": float(efficiency),
                    "final_score": float(final_score),
                }
            )

        ranked.sort(
            key=lambda c: (
                -c["final_score"],
                -c["sbert_score_norm"],
                str(c["id"]),
            )
        )
    else:
        ranked = []

    ranked_ids = [str(c["id"]) for c in ranked]

    # Ranking metrics
    h5 = hit_at_k(ranked_ids, relevant_ids, 5)
    h10 = hit_at_k(ranked_ids, relevant_ids, 10)

    p5 = precision_at_k(ranked_ids, relevant_ids, 5)
    p10 = precision_at_k(ranked_ids, relevant_ids, 10)

    r5 = recall_at_k(ranked_ids, relevant_ids, 5)
    r10 = recall_at_k(ranked_ids, relevant_ids, 10)

    rr = reciprocal_rank(ranked_ids, relevant_ids)

    # Best relevant rank
    best_rank = None
    best_title = None

    for rank, c in enumerate(ranked, start=1):
        if str(c["id"]) in relevant_ids:
            best_rank = rank
            best_title = c["title"]
            break

    # Number of relevant IDs recovered
    relevant_in_candidate = sum(1 for rid in candidate_ids if rid in relevant_ids)
    relevant_in_top5 = sum(1 for rid in ranked_ids[:5] if rid in relevant_ids)
    relevant_in_top10 = sum(1 for rid in ranked_ids[:10] if rid in relevant_ids)

    rows.append(
        {
            "scraped_title": row["scraped_title"],
            "query_text": query_text,
            "n_relevant_ids": len(relevant_ids),
            "relevant_ids": " | ".join(sorted(relevant_ids)),
            "candidate_pool_size": len(candidate_ids),
            "relevant_in_candidate_pool": relevant_in_candidate,
            "candidate_hit": cand_hit,
            "candidate_recall": cand_recall,
            "relevant_in_top5": relevant_in_top5,
            "relevant_in_top10": relevant_in_top10,
            "hit_at_5": h5,
            "hit_at_10": h10,
            "precision_at_5": p5,
            "precision_at_10": p10,
            "recall_at_5": r5,
            "recall_at_10": r10,
            "reciprocal_rank": rr,
            "best_relevant_rank": best_rank,
            "best_relevant_title": best_title,
            "top10_result_ids": " | ".join(ranked_ids[:10]),
            "top10_result_titles": " | ".join(str(c["title"]) for c in ranked[:10]),
        }
    )


detail_df = pd.DataFrame(rows)

# ============================================================
# 5. Macro-average metrics across queries
# ============================================================
summary = {
    "n_query": len(detail_df),
    "candidate_hit_rate": float(detail_df["candidate_hit"].mean()),
    "candidate_hit_count": int(detail_df["candidate_hit"].sum()),
    "candidate_recall_macro": float(detail_df["candidate_recall"].mean()),
    "hit_at_5": float(detail_df["hit_at_5"].mean()),
    "hit_at_5_count": int(detail_df["hit_at_5"].sum()),
    "hit_at_10": float(detail_df["hit_at_10"].mean()),
    "hit_at_10_count": int(detail_df["hit_at_10"].sum()),
    "precision_at_5_macro": float(detail_df["precision_at_5"].mean()),
    "precision_at_10_macro": float(detail_df["precision_at_10"].mean()),
    "recall_at_5_macro": float(detail_df["recall_at_5"].mean()),
    "recall_at_10_macro": float(detail_df["recall_at_10"].mean()),
    "mrr": float(detail_df["reciprocal_rank"].mean()),
    "semantic_top_k": TOP_K_SBERT,
    "keyword_extra_max": MAX_KEYWORD_EXTRA,
    "semantic_mode": "minmax",
    "w1": W1_FINAL,
    "w2": W2_FINAL,
}

summary_df = pd.DataFrame([summary])

# ============================================================
# 6. Save
# ============================================================
summary_df.to_csv(
    SUMMARY_OUT,
    index=False,
    encoding="utf-8-sig",
)

detail_df.to_csv(
    DETAIL_OUT,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# 7. Print result
# ============================================================
print("\n" + "=" * 104)
print("HASIL FINAL TEST - INDEPENDENT GROUND TRUTH + PRECISION/RECALL")
print("=" * 104)

print(
    f"Candidate Hit Rate : "
    f"{summary['candidate_hit_rate']:.4f} "
    f"({summary['candidate_hit_count']}/{summary['n_query']})"
)

print(f"Candidate Recall   : {summary['candidate_recall_macro']:.4f}")

print(
    f"Hit@5              : "
    f"{summary['hit_at_5']:.4f} "
    f"({summary['hit_at_5_count']}/{summary['n_query']})"
)

print(
    f"Hit@10             : "
    f"{summary['hit_at_10']:.4f} "
    f"({summary['hit_at_10_count']}/{summary['n_query']})"
)

print(f"Precision@5        : {summary['precision_at_5_macro']:.4f}")

print(f"Precision@10       : {summary['precision_at_10_macro']:.4f}")

print(f"Recall@5           : {summary['recall_at_5_macro']:.4f}")

print(f"Recall@10          : {summary['recall_at_10_macro']:.4f}")

print(f"MRR                : {summary['mrr']:.4f}")

print("\nINTERPRETASI")
print(
    "- Candidate Recall = rata-rata proporsi relevant IDs yang berhasil "
    "masuk candidate pool."
)
print("- Precision@K = rata-rata proporsi hasil Top-K yang termasuk relevance set.")
print(
    "- Recall@K = rata-rata proporsi relevance set yang berhasil ditemukan pada Top-K."
)
print("- Hit@K hanya memeriksa apakah minimal satu relevant ID ditemukan.")
print("- MRR mengukur posisi relevant ID pertama.")

print("\nCATATAN METODOLOGIS")
print(
    "- Precision/Recall di sini dihitung terhadap relevance set yang "
    "telah ditetapkan, bukan seluruh resep relevan potensial di corpus."
)
print("- Konfigurasi tetap frozen; notebook ini hanya menambahkan metrik.")

print("\nOUTPUT")
print(f"- {SUMMARY_OUT}")
print(f"- {DETAIL_OUT}")

c:\Users\Mayceren\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


VALIDASI TEST SET
Jumlah query              : 90
Duplicate query+relevance : 0


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6179.42it/s]



FROZEN CONFIG
Semantic top-k            : 50
Keyword extra max         : 20
WLC                       : 0.7 semantic + 0.3 efficiency
Semantic normalization    : min-max
Efficiency                : harmonic mean

GROUND TRUTH VALIDATION
Unique relevant IDs          : 160
Relevant IDs found in Chroma : 160

Menjalankan frozen configuration pada test set...

HASIL FINAL TEST - INDEPENDENT GROUND TRUTH + PRECISION/RECALL
Candidate Hit Rate : 0.6000 (54/90)
Candidate Recall   : 0.5555
Hit@5              : 0.2111 (19/90)
Hit@10             : 0.3000 (27/90)
Precision@5        : 0.0467
Precision@10       : 0.0411
Recall@5           : 0.1704
Recall@10          : 0.2593
MRR                : 0.1485

INTERPRETASI
- Candidate Recall = rata-rata proporsi relevant IDs yang berhasil masuk candidate pool.
- Precision@K = rata-rata proporsi hasil Top-K yang termasuk relevance set.
- Recall@K = rata-rata proporsi relevance set yang berhasil ditemukan pada Top-K.
- Hit@K hanya memeriksa apakah minimal s